In [1]:
!pip install mpi4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.7 MB/s eta 0:00:00


In [2]:
from mpi4py import MPI
import random
import time

# Sequential Quicksort
def quicksort(arr):

    if len(arr) <= 1:
        return arr

    pivot = arr[len(arr) // 2]

    left = [x for x in arr if x < pivot]

    middle = [x for x in arr if x == pivot]

    right = [x for x in arr if x > pivot]

    return quicksort(left) + middle + quicksort(right)

# MPI Setup
comm = MPI.COMM_WORLD

rank = comm.Get_rank()

size = comm.Get_size()

# Root Process Creates Data
if rank == 0:

    N = 100000

    data = [random.randint(1, 1000000) for _ in range(N)]

    start = time.time()

else:

    data = None

# Scatter Data
local_data = comm.scatter(
    [data[i::size] for i in range(size)] if rank == 0 else None,
    root=0
)

# Local Sorting
local_sorted = quicksort(local_data)

# Gather Sorted Chunks
gathered = comm.gather(local_sorted, root=0)

# Final Merge at Root
if rank == 0:

    merged = []

    for chunk in gathered:
        merged.extend(chunk)

    final_sorted = quicksort(merged)

    end = time.time()

    print("First 20 Sorted Elements:")
    print(final_sorted[:20])

    print("\nExecution Time:", end - start, "seconds")

First 20 Sorted Elements:
[6, 6, 14, 24, 27, 46, 47, 85, 88, 91, 97, 106, 116, 138, 141, 162, 171, 188, 219, 236]

Execution Time: 0.33681178092956543 seconds
